## Orbital Debris Exploration

Initial exploration of the Orbital Debris SQL database previously created. Contains various panda queries, sql queries, and visualizations to really 'dig' into the dataset.  This will help us narrow down and refine our primary and secondary questions, as well as help us polish the sql queries and chart visualizations for our primary and secondary questions.  Many of these query results can and likely will be used for supporting visuals to add context to our primary and secondary questions.  You may be able to tell a story in 3 visuals, but I am betting that story will be a broken story at best.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import utility as utils
import sqlite3 as sql
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display, Markdown 

pd.set_option('display.max_columns', None)

# Initialize Visualization Settings
# Set the Palette (High Contrast Neon)
sns.set_palette("plasma") 

# Set the Background (Deep Space Black)
plt.style.use('dark_background')

plt.rcParams.update({
    "grid.alpha": 0.2,            # Faint, non-intrusive grid
    "axes.facecolor": "black",    # No grey haze in plot area
    "figure.facecolor": "black",  # No grey haze in outer area
    "text.color": "white",        # High contrast text
    "axes.labelcolor": "white",   # Axis labels
    "xtick.color": "white",       # Ticks
    "ytick.color": "white",
    "legend.facecolor": "black",  # Legend background
    "legend.edgecolor": "white"   # Legend border
})

print(plt.colormaps())

orbital_debris_conn = sql.connect('../data/clean/orbital_debris.db')

# Load kinetic_master as a pandas DataFrame for any eda we want to do using just pandas. We 
# could also load each table as a separate DataFrame, but this is easier for now and I can 
# use SQL queries against the proper database connection if I want to do more complex queries.
# Loading master_df just allows me to do quick things with pandas without having to write the
# SQL query logic each time.
master_df = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)

display(utils.query_all_satellites(orbital_debris_conn))

### Payload Vulnerability Report (Pre-Zombie Check)
 
**The issue:**
A satellite’s operational status only tells us if it’s working right now—not whether it’s past its expected design life. Satellites that are still "operational" but older than 15 years are at higher risk of sudden failure and loss of control.

**What we do:**
- Identify all in-orbit payloads marked as `OPERATIONAL`.
- Flag those that are older than 15 years as "vulnerable" assets.
- Report the total number and share of these high-risk payloads, and break down the results by orbit class.

**Why it matters:**
Spotting aging, at-risk satellites helps us understand the true health of the active fleet and anticipate future risks. This is a first-pass estimate; a more detailed analysis will use actual design-life data in the next pipeline stage.

In [ ]:
# Define the Vulnerability Threshold (Industry Standard: 5 - 15 Years)
vulnerability_threshold = 15

# Isolate Operational Payloads
op_payloads_mask = (master_df['in_orbit'] == 1) & \
                   (master_df['object_type'] == 'PAYLOAD') & \
                   (master_df['ops_status'] == 'OPERATIONAL')

op_payloads = master_df[op_payloads_mask].copy()

# Calculate Vulnerability
total_op = len(op_payloads)
vulnerable_op = op_payloads[op_payloads['sat_age_years'] > vulnerability_threshold]
vulnerable_count = len(vulnerable_op)

print(f"{'--- PAYLOAD VULNERABILITY AUDIT ---':^55}")
print(f"Total Operational Payloads:     {total_op:,}")
print(f"Aged Assets (>15 Years):        {vulnerable_count:,}")
print(f"Vulnerability Rate:             {(vulnerable_count/total_op if total_op > 0 else 0):.1%}")
print("-" * 55)

# Breakdown by Orbit Class
if vulnerable_count > 0:
    print("\nVulnerability Distribution by Orbit:")
    print(vulnerable_op['orbit_class'].value_counts())

print(f"\n👴 {vulnerable_count} high-risk active assets identified 👴")